In [ ]:
# Cell 1: Configure RFUAV lightweight spectrogram CNN experiment
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import yaml
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Conv2D, Dense, Dropout, GlobalAveragePooling2D, Input, MaxPooling2D
from tensorflow.keras.models import Model, load_model

notebook_dir = Path().resolve()
if notebook_dir.name == 'notebooks':
    project_root = notebook_dir.parent
elif (notebook_dir / 'notebooks').exists() and (notebook_dir / 'src').exists():
    project_root = notebook_dir
elif (notebook_dir / 'ML-wireless-signal-classification').exists():
    project_root = notebook_dir / 'ML-wireless-signal-classification'
else:
    project_root = notebook_dir

cfg_path = project_root / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    local_cfg = yaml.safe_load(cfg_path.read_text()) or {}
    dcfg = local_cfg.get('datasets', {}).get('rfuav', {}) or {}
    data_dir = Path(dcfg.get('data_dir', Path(local_cfg.get('dataset_root', '/scratch/rameyjm7/datasets')) / 'RFUAV'))
    archive_dir = Path(dcfg.get('archive_dir', data_dir / 'archives'))
    extract_root = Path(dcfg.get('extract_dir', data_dir / 'extracted'))
else:
    data_dir = Path('/scratch/rameyjm7/datasets/RFUAV')
    archive_dir = data_dir / 'archives'
    extract_root = data_dir / 'extracted'

SCRATCH_ROOT = Path(os.getenv('RFUAV_SCRATCH_ROOT', '/scratch/rameyjm7/ML-wireless-signal-classification'))
outputs_dir = project_root / 'outputs' / 'rfuav_eval'
model_dir = project_root / 'models' / 'rfuav'
archive_dir = Path(os.getenv('RFUAV_ARCHIVE_DIR', str(archive_dir)))
extract_root = Path(os.getenv('RFUAV_EXTRACT_ROOT', str(extract_root)))
cache_dir = Path(os.getenv('RFUAV_SPEC_CACHE_DIR', str(SCRATCH_ROOT / 'cache' / 'rfuav' / 'spectrogram_cnn_cache')))
manifest_path = Path(os.getenv('RFUAV_MANIFEST_PATH', str(SCRATCH_ROOT / 'manifests' / 'rfuav_manifest.csv')))

for path in (outputs_dir, model_dir, extract_root, cache_dir, manifest_path.parent):
    path.mkdir(parents=True, exist_ok=True)

MAX_IQ_SAMPLES = int(os.getenv('RFUAV_MAX_IQ_SAMPLES', '65536'))
DATA_FRACTION = float(os.getenv('RFUAV_DATA_FRACTION', '1.0'))
BATCH_SIZE = int(os.getenv('RFUAV_BATCH_SIZE', '32'))
SHUFFLE_BUFFER = int(os.getenv('RFUAV_SHUFFLE_BUFFER', '256'))
SPEC_NFFT = int(os.getenv('RFUAV_SPEC_NFFT', '256'))
SPEC_HOP = int(os.getenv('RFUAV_SPEC_HOP', '128'))
SPEC_TIME_BINS = int(os.getenv('RFUAV_SPEC_TIME_BINS', '256'))
SPEC_EVAL_WINDOWS = int(os.getenv('RFUAV_SPEC_EVAL_WINDOWS', '1'))
CNN_WEIGHT_DECAY = float(os.getenv('RFUAV_CNN_WEIGHT_DECAY', '1e-5'))
EVAL_LIMIT = int(os.getenv('RFUAV_EVAL_LIMIT', '0'))
RFUAV_RAW_DTYPE = np.float32  # Author tooling reads raw datapacks as interleaved float32 I/Q.
RFUAV_SAMPLE_RATE_HZ = float(os.getenv('RFUAV_SAMPLE_RATE_HZ', '100e6'))
RANDOM_STATE = 3407

print('Project root:', project_root)
print('RFUAV dataset:', data_dir)
print('Archive dir:', archive_dir)
print('Archive extraction root:', extract_root)
print('Spectrogram cache:', cache_dir)
print('Manifest:', manifest_path)
print('Light spectrogram bins:', SPEC_NFFT, 'x', SPEC_TIME_BINS)
print('Raw dtype:', RFUAV_RAW_DTYPE, 'Sample rate:', RFUAV_SAMPLE_RATE_HZ)
print('Data fraction:', DATA_FRACTION)
assert data_dir.exists(), f'Missing RFUAV dataset directory: {data_dir}'
assert archive_dir.exists(), f'Missing RFUAV archive directory: {archive_dir}. Run notebook 10 Cell 6 first.'

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
try:
    gpus = tf.config.list_physical_devices('GPU')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('GPUs:', gpus)
except Exception as exc:
    print('GPU memory-growth setup skipped:', repr(exc))

archive_paths = sorted(archive_dir.glob('*.rar'))
print('RFUAV class archives found:', len(archive_paths))
for archive in archive_paths[:10]:
    print(' -', archive.name)


In [ ]:
# Cell 2: Build or load an RFUAV manifest by extracting archive contents to scratch
IQ_SUFFIXES = {'.npy', '.npz', '.mat', '.csv', '.bin', '.dat', '.iq', '.complex', '.txt'}
META_SUFFIXES = {'.md', '.pdf', '.png', '.jpg', '.jpeg', '.json', '.yaml', '.yml', '.xml'}


def safe_label_from_archive(path: Path) -> str:
    label = path.stem.strip()
    label = re.sub(r'\s+', '_', label)
    label = re.sub(r'[^A-Za-z0-9_\-]+', '', label)
    return label or 'unknown'


def extractor_command(archive_path: Path, extract_to: Path):
    if shutil.which('unrar'):
        return ['unrar', 'x', '-o+', str(archive_path), str(extract_to) + '/']
    if shutil.which('bsdtar'):
        return ['bsdtar', '-xf', str(archive_path), '-C', str(extract_to)]
    if shutil.which('7z'):
        return ['7z', 'x', '-y', f'-o{extract_to}', str(archive_path)]
    raise RuntimeError('No RAR extractor found. Install one of: unrar, bsdtar/libarchive, or p7zip/7z.')


def extract_archive(path: Path, target: Path):
    done = target / '.extract_done'
    done_from_notebook10 = target / '.extract_complete'
    if (done.exists() or done_from_notebook10.exists()) and any(p.is_file() for p in target.rglob('*')):
        return
    target.mkdir(parents=True, exist_ok=True)
    print('Extracting:', path.name, '->', target)
    subprocess.run(extractor_command(path, target), check=True)
    done.write_text(datetime.now(timezone.utc).isoformat(), encoding='utf-8')


def discover_iq_files(root: Path) -> list[Path]:
    files = []
    for file_path in root.rglob('*'):
        if not file_path.is_file():
            continue
        suffix = file_path.suffix.lower()
        if suffix in META_SUFFIXES:
            continue
        if suffix in IQ_SUFFIXES:
            files.append(file_path)
    return sorted(files)

if manifest_path.exists() and os.getenv('RFUAV_REBUILD_MANIFEST', '0').lower() not in {'1', 'true', 'yes'}:
    manifest_df = pd.read_csv(manifest_path)
    print('Loaded manifest:', manifest_path, 'rows:', len(manifest_df))
else:
    rows = []
    if not archive_paths:
        raise FileNotFoundError(f'No .rar archives found in {data_dir}. Run notebook 10 Cell 6 first.')
    for archive in list(reversed(archive_paths)):
        label = safe_label_from_archive(archive)
        class_extract_dir = extract_root / label
        extract_archive(archive, class_extract_dir)
        iq_files = discover_iq_files(class_extract_dir)
        print(label, 'files:', len(iq_files))
        for file_path in iq_files:
            rel = file_path.relative_to(class_extract_dir)
            rows.append({
                'filepath': str(file_path),
                'label': label,
                'archive': archive.name,
                'relative_path': str(rel),
                'size_bytes': int(file_path.stat().st_size),
            })
    manifest_df = pd.DataFrame(rows)
    if manifest_df.empty:
        raise RuntimeError('RFUAV archives extracted, but no candidate IQ files were discovered. Inspect extracted files under extract_root.')
    manifest_df.to_csv(manifest_path, index=False)
    print('Saved manifest:', manifest_path)


def repair_or_drop_missing_files(frame: pd.DataFrame) -> pd.DataFrame:
    # Cached manifests can outlive scratch cleanup or extraction layout changes.
    # Repair rows by filename when possible; otherwise drop them before TensorFlow sees them.
    repaired = frame.copy()
    missing_rows = []
    repaired_count = 0
    search_cache = {}
    for idx, value in repaired['filepath'].items():
        path = Path(value)
        if path.exists():
            continue
        filename = path.name
        matches = search_cache.get(filename)
        if matches is None:
            matches = sorted(extract_root.rglob(filename)) if extract_root.exists() else []
            search_cache[filename] = matches
        if matches:
            repaired.at[idx, 'filepath'] = str(matches[0])
            repaired_count += 1
        else:
            missing_rows.append(idx)
    if missing_rows:
        print(f'Dropping {len(missing_rows)} manifest rows with missing files.')
        repaired = repaired.drop(index=missing_rows)
    if repaired_count:
        print(f'Repaired {repaired_count} stale manifest paths by filename search.')
    repaired = repaired.reset_index(drop=True)
    if repaired.empty:
        raise FileNotFoundError('All RFUAV manifest rows point to missing files. Re-run notebook 10 extraction or rebuild the manifest.')
    return repaired

manifest_df = repair_or_drop_missing_files(manifest_df)

# Keep only actual IQ-like payloads. RFUAV ships pack*.xml metadata next to .iq files;
# treating XML as raw IQ causes model collapse.
before_iq_filter = len(manifest_df)
manifest_df = manifest_df[manifest_df['filepath'].map(lambda value: Path(value).suffix.lower() in IQ_SUFFIXES)].reset_index(drop=True)
dropped_non_iq = before_iq_filter - len(manifest_df)
if dropped_non_iq:
    print(f'Dropped {dropped_non_iq} non-IQ manifest rows before training.')
if manifest_df.empty:
    raise RuntimeError('RFUAV manifest has no IQ payload rows after filtering. Run notebook 10 Cell 7 and rebuild manifest.')

label_names = sorted(manifest_df['label'].unique().tolist())
label_to_idx = {name: idx for idx, name in enumerate(label_names)}
manifest_df['label_idx'] = manifest_df['label'].map(label_to_idx).astype(np.int64)
num_classes = len(label_names)
print('classes:', num_classes)
print(manifest_df.groupby('label').size().sort_values(ascending=False).head(40))


In [ ]:
# Cell 3: Define RFUAV IQ loading, fast spectrogram caching, and datasets
def _read_numeric_text(path: Path) -> np.ndarray:
    try:
        arr = np.loadtxt(path, delimiter=',')
    except Exception:
        arr = np.loadtxt(path)
    return np.asarray(arr)


def load_iq_file(path_like) -> np.ndarray:
    path = Path(path_like)
    suffix = path.suffix.lower()
    if suffix == '.npy':
        arr = np.load(path, allow_pickle=True)
    elif suffix == '.npz':
        obj = np.load(path, allow_pickle=True)
        key = 'x' if 'x' in obj else ('iq' if 'iq' in obj else obj.files[0])
        arr = obj[key]
    elif suffix == '.mat':
        import scipy.io as sio
        try:
            mat = sio.loadmat(path)
            keys = [k for k in mat.keys() if not k.startswith('__')]
            if not keys:
                raise ValueError(f'No arrays in mat file: {path}')
            arr = mat[keys[0]]
        except NotImplementedError:
            import h5py
            with h5py.File(path, 'r') as h5f:
                key = next(iter(h5f.keys()))
                arr = np.array(h5f[key])
    elif suffix in {'.csv', '.txt'}:
        arr = _read_numeric_text(path)
    else:
        # RFUAV author tooling uses np.fromfile(..., dtype=np.float32), then I + 1j*Q.
        raw = np.fromfile(path, dtype=RFUAV_RAW_DTYPE)
        if raw.size < 4 or not np.isfinite(raw).all():
            raw = np.fromfile(path, dtype=np.int16).astype(np.float32)
        arr = raw

    arr = np.asarray(arr)
    arr = np.squeeze(arr)
    if np.iscomplexobj(arr):
        iq = np.stack([arr.real, arr.imag], axis=-1)
    elif arr.ndim == 1:
        if arr.size % 2 != 0:
            arr = arr[:-1]
        iq = arr.reshape(-1, 2)
    elif arr.ndim >= 2 and 2 in arr.shape:
        if arr.shape[-1] == 2:
            iq = arr.reshape(-1, 2)
        else:
            iq = np.moveaxis(arr, list(arr.shape).index(2), -1).reshape(-1, 2)
    else:
        flat = arr.reshape(-1)
        if flat.size % 2 != 0:
            flat = flat[:-1]
        iq = flat.reshape(-1, 2)
    iq = np.asarray(iq, dtype=np.float32)
    iq = iq[np.isfinite(iq).all(axis=1)]
    if iq.shape[0] < SPEC_NFFT:
        iq = np.pad(iq, ((0, SPEC_NFFT - iq.shape[0]), (0, 0)), mode='constant')
    return iq


def normalize_iq_window(out: np.ndarray) -> np.ndarray:
    scale = np.sqrt(np.mean(out.astype(np.float32) ** 2) + 1e-8)
    return (out / scale).astype(np.float32)


def select_active_window(iq: np.ndarray, max_samples=MAX_IQ_SAMPLES) -> np.ndarray:
    if iq.shape[0] <= max_samples:
        return normalize_iq_window(iq)
    power = np.sum(iq.astype(np.float32) ** 2, axis=1)
    kernel = np.ones(min(8192, max_samples), dtype=np.float32)
    smooth = np.convolve(power, kernel / kernel.size, mode='same')
    center = int(np.argmax(smooth))
    start = max(0, min(center - max_samples // 2, iq.shape[0] - max_samples))
    return normalize_iq_window(iq[start:start + max_samples])


def select_seeded_window(iq: np.ndarray, filepath: str, view_idx: int, max_samples=MAX_IQ_SAMPLES) -> np.ndarray:
    if view_idx == 0:
        return select_active_window(iq, max_samples=max_samples)
    if iq.shape[0] <= max_samples:
        return normalize_iq_window(iq)
    seed_token = f'{filepath}|{view_idx}|{RANDOM_STATE}'
    seed = int(hashlib.sha1(seed_token.encode('utf-8')).hexdigest()[:8], 16)
    rng = np.random.default_rng(seed)
    start = int(rng.integers(0, iq.shape[0] - max_samples + 1))
    return normalize_iq_window(iq[start:start + max_samples])


def cache_key(filepath: str) -> str:
    path = Path(filepath)
    stat = path.stat()
    token = f'{path}|{stat.st_size}|{stat.st_mtime_ns}|{MAX_IQ_SAMPLES}|{SPEC_NFFT}|{SPEC_HOP}|{SPEC_TIME_BINS}'
    return hashlib.sha1(token.encode('utf-8')).hexdigest()


def safe_load_cache(path: Path):
    try:
        with np.load(path) as data:
            arr = np.asarray(data['x'], dtype=np.float32)
        if arr.shape == input_shape and np.isfinite(arr).all():
            return arr
    except Exception:
        path.unlink(missing_ok=True)
    return None


def write_cache_atomic(path: Path, arr: np.ndarray):
    tmp = path.with_suffix(path.suffix + f'.{os.getpid()}.tmp')
    with tmp.open('wb') as f:
        np.savez_compressed(f, x=arr.astype(np.float32))
    tmp.replace(path)


def complex_spectrogram(iq: np.ndarray) -> np.ndarray:
    # Lightweight 33c-style representation: real/imag STFT channels only.
    complex_iq = iq[:, 0].astype(np.float32) + 1j * iq[:, 1].astype(np.float32)
    if complex_iq.size < SPEC_NFFT:
        complex_iq = np.pad(complex_iq, (0, SPEC_NFFT - complex_iq.size), mode='constant')
    starts = np.arange(0, max(1, complex_iq.size - SPEC_NFFT + 1), SPEC_HOP, dtype=np.int64)
    if starts.size == 0:
        starts = np.array([0], dtype=np.int64)
    if starts.size >= SPEC_TIME_BINS:
        idx = np.linspace(0, starts.size - 1, SPEC_TIME_BINS).round().astype(np.int64)
        starts = starts[idx]
    else:
        starts = np.pad(starts, (0, SPEC_TIME_BINS - starts.size), mode='edge')
    window = np.hanning(SPEC_NFFT).astype(np.float32)
    frames = np.stack([complex_iq[start:start + SPEC_NFFT] * window for start in starts], axis=0)
    spec = np.fft.fftshift(np.fft.fft(frames, axis=1), axes=1) / float(SPEC_NFFT)
    out = np.stack([spec.real.astype(np.float32), spec.imag.astype(np.float32)], axis=-1)
    out = out / (np.std(out) + 1e-6)
    return np.clip(out, -6.0, 6.0).astype(np.float32)

input_shape = (SPEC_TIME_BINS, SPEC_NFFT, 2)




def load_precomputed_spectrogram(path_like) -> np.ndarray:
    with np.load(Path(path_like)) as data:
        arr = np.asarray(data['x'], dtype=np.float32)
    if arr.shape != input_shape:
        raise ValueError(f'Bad precomputed spectrogram shape {arr.shape}; expected {input_shape}: {path_like}')
    return arr


def prepare_spectrogram(filepath: str) -> np.ndarray:
    cache_path = cache_dir / f'{cache_key(filepath)}.npz'
    if cache_path.exists():
        cached = safe_load_cache(cache_path)
        if cached is not None:
            return cached
    iq = select_active_window(load_iq_file(filepath))
    spec = complex_spectrogram(iq)
    write_cache_atomic(cache_path, spec)
    return spec

if DATA_FRACTION < 1.0:
    manifest_df = (
        manifest_df.groupby('label_idx', group_keys=False)
        .apply(lambda frame: frame.sample(max(1, int(math.ceil(len(frame) * DATA_FRACTION))), random_state=RANDOM_STATE))
        .reset_index(drop=True)
    )
    print('Fractioned rows:', len(manifest_df))

def split_by_class(frame: pd.DataFrame, train_frac=0.70, val_frac=0.15, random_state=RANDOM_STATE):
    # sklearn's second stratified split fails when tmp has only one sample for a class.
    # This deterministic per-class split keeps rare classes usable and avoids dropping data.
    rng = np.random.default_rng(random_state)
    train_parts = []
    val_parts = []
    test_parts = []
    for label_idx, group in frame.groupby('label_idx', sort=True):
        group = group.sample(frac=1.0, random_state=random_state + int(label_idx)).reset_index(drop=True)
        n = len(group)
        if n < 3:
            # Too small for all splits; keep it in train and let balanced replay upsample it.
            train_parts.append(group)
            print(f'Class {label_idx} has only {n} sample(s); assigning all to train.')
            continue
        n_train = max(1, int(round(n * train_frac)))
        n_val = max(1, int(round(n * val_frac)))
        if n_train + n_val >= n:
            n_train = max(1, n - 2)
            n_val = 1
        train_parts.append(group.iloc[:n_train])
        val_parts.append(group.iloc[n_train:n_train + n_val])
        test_parts.append(group.iloc[n_train + n_val:])
    train = pd.concat(train_parts, ignore_index=True).sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    val = pd.concat(val_parts, ignore_index=True).sample(frac=1.0, random_state=random_state).reset_index(drop=True) if val_parts else pd.DataFrame(columns=frame.columns)
    test = pd.concat(test_parts, ignore_index=True).sample(frac=1.0, random_state=random_state).reset_index(drop=True) if test_parts else pd.DataFrame(columns=frame.columns)
    if val.empty or test.empty:
        raise ValueError('RFUAV split produced an empty val/test split. Increase DATA_FRACTION or add more RFUAV samples.')
    return train, val, test

train_df, val_df, test_df = split_by_class(manifest_df)
print('train/val/test:', len(train_df), len(val_df), len(test_df))
print('val class counts:', val_df['label_idx'].value_counts().sort_index().to_dict())
print('test class counts:', test_df['label_idx'].value_counts().sort_index().to_dict())

class_counts = train_df['label_idx'].value_counts().sort_index()
max_count = int(class_counts.max())
balanced_train_df = pd.concat([
    train_df[train_df['label_idx'] == label_idx].sample(max_count, replace=True, random_state=RANDOM_STATE + int(label_idx))
    for label_idx in class_counts.index
], ignore_index=True).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
print('balanced train rows:', len(balanced_train_df))


def make_dataset(frame: pd.DataFrame, training=False):
    paths = frame['filepath'].astype(str).to_numpy()
    labels = frame['label_idx'].to_numpy(dtype=np.int64)

    def gen():
        for filepath, label in zip(paths, labels):
            yield prepare_spectrogram(filepath), np.int64(label)

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=input_shape, dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int64),
        ),
    )
    if training:
        ds = ds.shuffle(min(SHUFFLE_BUFFER, len(frame)), reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(balanced_train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)
train_steps = int(math.ceil(len(balanced_train_df) / BATCH_SIZE))
validation_steps = int(math.ceil(len(val_df) / BATCH_SIZE))
print('steps:', train_steps, validation_steps)


In [ ]:
# Cell 4: Precompute compact RFUAV spectrogram tensors for faster training
# First run does raw IQ read + FFT work. Later training/eval cells load compact tensors directly.
precompute_dir = Path(os.getenv('RFUAV_PRECOMPUTE_DIR', str(cache_dir / 'precomputed_tensors')))
precompute_dir.mkdir(parents=True, exist_ok=True)
RFUAV_PRECOMPUTE_AUGMENTS = int(os.getenv('RFUAV_PRECOMPUTE_AUGMENTS', '2'))
RFUAV_FORCE_PRECOMPUTE = os.getenv('RFUAV_FORCE_PRECOMPUTE', '0').lower() in {'1', 'true', 'yes'}
RFUAV_COMPRESS_PRECOMPUTE = os.getenv('RFUAV_COMPRESS_PRECOMPUTE', '0').lower() in {'1', 'true', 'yes'}


def precompute_key(filepath: str, split: str, view_idx: int) -> str:
    path = Path(filepath)
    stat = path.stat()
    token = '|'.join([
        str(path), str(stat.st_size), str(stat.st_mtime_ns), split, str(view_idx),
        str(MAX_IQ_SAMPLES), str(SPEC_NFFT), str(SPEC_HOP), str(SPEC_TIME_BINS), 'v1',
    ])
    return hashlib.sha1(token.encode('utf-8')).hexdigest()


def save_precomputed(path: Path, spec: np.ndarray):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f'.{os.getpid()}.tmp')
    with tmp.open('wb') as handle:
        if RFUAV_COMPRESS_PRECOMPUTE:
            np.savez_compressed(handle, x=spec.astype(np.float32))
        else:
            np.savez(handle, x=spec.astype(np.float32))
    tmp.replace(path)


def valid_precomputed(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        with np.load(path) as data:
            arr = np.asarray(data['x'], dtype=np.float32)
        return arr.shape == input_shape and np.isfinite(arr).all()
    except Exception:
        path.unlink(missing_ok=True)
        return False


def precompute_frame(frame: pd.DataFrame, split: str, augment_count: int) -> pd.DataFrame:
    rows = []
    total = len(frame)
    for row_idx, row in enumerate(frame.itertuples(index=False), start=1):
        filepath = str(row.filepath)
        label_idx = int(row.label_idx)
        iq = None
        for view_idx in range(max(1, augment_count)):
            out_path = precompute_dir / split / f'{precompute_key(filepath, split, view_idx)}.npz'
            if RFUAV_FORCE_PRECOMPUTE or not valid_precomputed(out_path):
                if iq is None:
                    iq = load_iq_file(filepath)
                spec = complex_spectrogram(select_seeded_window(iq, filepath, view_idx))
                save_precomputed(out_path, spec)
            rows.append({
                'spec_path': str(out_path),
                'source_filepath': filepath,
                'label_idx': label_idx,
                'label': getattr(row, 'label', label_names[label_idx]),
                'split': split,
                'view_idx': view_idx,
            })
        if row_idx <= 5 or row_idx % 25 == 0 or row_idx == total:
            print(f'Precomputed {split}: {row_idx}/{total} raw files -> {len(rows)} tensor rows', flush=True)
    result = pd.DataFrame(rows)
    manifest_out = precompute_dir / f'{split}_manifest.csv'
    result.to_csv(manifest_out, index=False)
    print(f'Saved {split} precompute manifest:', manifest_out, 'rows:', len(result))
    return result


train_source_df = balanced_train_df if os.getenv('RFUAV_TRAIN_ON_BALANCED_REPLAY', '1').lower() not in {'0', 'false', 'no'} else train_df
precomputed_train_df = precompute_frame(train_source_df, 'train', RFUAV_PRECOMPUTE_AUGMENTS)
precomputed_val_df = precompute_frame(val_df, 'val', 1)
precomputed_test_df = precompute_frame(test_df, 'test', 1)

print('Precomputed train class counts:', precomputed_train_df['label_idx'].value_counts().sort_index().to_dict())
print('Precomputed val class counts:', precomputed_val_df['label_idx'].value_counts().sort_index().to_dict())
print('Precomputed test class counts:', precomputed_test_df['label_idx'].value_counts().sort_index().to_dict())


In [ ]:
# Cell 5: Build or resume the RFUAV lightweight spectrogram CNN model
def conv_block(x, filters, dropout=0.0):
    x = Conv2D(
        filters,
        5,
        padding='same',
        use_bias=False,
        kernel_regularizer=tf.keras.regularizers.l2(CNN_WEIGHT_DECAY),
    )(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    if dropout > 0:
        x = Dropout(dropout)(x)
    return x


def build_cnn_model():
    inputs = Input(shape=input_shape)
    x = conv_block(inputs, 32, dropout=0.05)
    x = conv_block(x, 64, dropout=0.10)
    x = conv_block(x, 96, dropout=0.15)
    x = conv_block(x, 128, dropout=0.20)
    x = GlobalAveragePooling2D()(x)
    x = Dense(192, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(CNN_WEIGHT_DECAY))(x)
    x = Dropout(0.35)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    return Model(inputs, outputs, name='rfuav_spectrogram_cnn')

best_path = model_dir / 'rfuav_spectrogram_cnn_best.keras'
final_path = model_dir / 'rfuav_spectrogram_cnn_final.keras'
history_path = model_dir / 'rfuav_spectrogram_cnn_history.json'
history_csv_path = model_dir / 'rfuav_spectrogram_cnn_training_history.csv'

resume_path = final_path if final_path.exists() else (best_path if best_path.exists() else None)
if resume_path is not None:
    print('Attempting resume from:', resume_path)
    try:
        model = load_model(resume_path, compile=False)
        if tuple(model.input_shape[1:]) != tuple(input_shape) or model.output_shape[-1] != num_classes:
            print('Checkpoint shape/classes mismatch; starting fresh.', model.input_shape, model.output_shape)
            model = build_cnn_model()
        else:
            print('Resuming compatible checkpoint.')
    except Exception as exc:
        print('Could not load checkpoint, starting fresh:', repr(exc))
        model = build_cnn_model()
else:
    model = build_cnn_model()

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=float(os.getenv('RFUAV_CNN_LR', '1e-4')),
        clipnorm=1.0,
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
    jit_compile=os.getenv('RFUAV_JIT_COMPILE', '0').lower() in {'1', 'true', 'yes'},
)
model.summary()


In [ ]:
# Cell 6: Train or continue RFUAV spectrogram CNN model and save cumulative history
# Fast-start knobs for RFUAV: cold spectrogram generation is expensive, so avoid filling a huge shuffle buffer before epoch 1.
RFUAV_FAST_START = os.getenv('RFUAV_FAST_START', '1').lower() not in {'0', 'false', 'no'}
RFUAV_CELL5_SHUFFLE_BUFFER = int(os.getenv('RFUAV_CELL5_SHUFFLE_BUFFER', '0' if RFUAV_FAST_START else str(SHUFFLE_BUFFER)))
RFUAV_TRAIN_ON_BALANCED_REPLAY = os.getenv('RFUAV_TRAIN_ON_BALANCED_REPLAY', '1').lower() not in {'0', 'false', 'no'}
RFUAV_VAL_METRIC_LIMIT_DEFAULT = '256' if RFUAV_FAST_START else '0'

def sanitize_existing_paths(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    frame = frame.copy().reset_index(drop=True)
    exists_mask = frame['filepath'].map(lambda value: Path(value).exists())
    missing_count = int((~exists_mask).sum())
    if missing_count:
        print(f'{name}: dropping {missing_count} missing files before dataset construction.')
    frame = frame[exists_mask].reset_index(drop=True)
    if frame.empty:
        raise FileNotFoundError(f'{name} has no existing RFUAV files after filtering. Re-run Cell 2 or notebook 10 extraction.')
    return frame

# Cell 5 may be rerun after scratch cleanup, so sanitize all active splits here too.
train_df = sanitize_existing_paths(train_df, 'train_df')
val_df = sanitize_existing_paths(val_df, 'val_df')
if 'test_df' in globals():
    test_df = sanitize_existing_paths(test_df, 'test_df')
if 'balanced_train_df' in globals():
    balanced_train_df = sanitize_existing_paths(balanced_train_df, 'balanced_train_df')

if 'precomputed_train_df' in globals() and 'precomputed_val_df' in globals():
    print('Using precomputed RFUAV spectrogram tensors for training.')
    train_frame_for_fit = precomputed_train_df.copy()
    val_frame_for_fit = precomputed_val_df.copy()
else:
    print('Precomputed tensors not found; falling back to raw IQ generator. Run Cell 4 for faster training.')
    train_frame_for_fit = balanced_train_df if RFUAV_TRAIN_ON_BALANCED_REPLAY else train_df
    val_frame_for_fit = val_df


def make_fit_dataset(frame: pd.DataFrame):
    use_precomputed = 'spec_path' in frame.columns
    paths = frame['spec_path' if use_precomputed else 'filepath'].astype(str).to_numpy()
    labels = frame['label_idx'].to_numpy(dtype=np.int64)

    def gen():
        for filepath, label in zip(paths, labels):
            if use_precomputed:
                yield load_precomputed_spectrogram(filepath), np.int64(label)
            else:
                yield prepare_spectrogram(filepath), np.int64(label)

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=input_shape, dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int64),
        ),
    )
    if RFUAV_CELL5_SHUFFLE_BUFFER > 1:
        ds = ds.shuffle(
            min(RFUAV_CELL5_SHUFFLE_BUFFER, len(frame)),
            seed=RANDOM_STATE,
            reshuffle_each_iteration=True,
        )
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Rebuild train/validation datasets here so Cell 5 can be made faster and stale-path safe without rerunning Cell 3.
train_ds = make_fit_dataset(train_frame_for_fit)
val_ds = make_fit_dataset(val_frame_for_fit)
train_steps = int(math.ceil(len(train_frame_for_fit) / BATCH_SIZE))
validation_steps = int(math.ceil(len(val_frame_for_fit) / BATCH_SIZE))
print('RFUAV Cell 5 fast start:', RFUAV_FAST_START)
print('RFUAV Cell 5 shuffle buffer:', RFUAV_CELL5_SHUFFLE_BUFFER)
print('RFUAV training rows used:', len(train_frame_for_fit), 'balanced replay:', RFUAV_TRAIN_ON_BALANCED_REPLAY)
print('RFUAV train steps:', train_steps)

# Recompile here so Cell 5 can recover from too-aggressive Cell 4 defaults without rerunning Cell 4.
# Early RFUAV batches can look collapsed with many classes; safer LR + no XLA makes debugging saner.
RFUAV_CELL5_LR = float(os.getenv('RFUAV_CELL5_LR', '5e-5'))
RFUAV_CELL5_CLIPNORM = float(os.getenv('RFUAV_CELL5_CLIPNORM', '1.0'))
try:
    loss_obj = tf.keras.losses.SparseCategoricalCrossentropy(
        label_smoothing=float(os.getenv('RFUAV_LABEL_SMOOTHING', '0.03'))
    )
except TypeError:
    # Older Keras builds do not expose label_smoothing for sparse CE.
    loss_obj = 'sparse_categorical_crossentropy'

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=RFUAV_CELL5_LR, clipnorm=RFUAV_CELL5_CLIPNORM),
    loss=loss_obj,
    metrics=['accuracy'],
    jit_compile=os.getenv('RFUAV_JIT_COMPILE', '0').lower() in {'1', 'true', 'yes'},
)
print('RFUAV Cell 5 learning rate:', RFUAV_CELL5_LR)
print('RFUAV Cell 5 loss:', loss_obj)

# Quick sanity check: verify labels are diverse and predictions are finite before training.
peek_x, peek_y = next(iter(train_ds.take(1)))
peek_probs = model.predict(peek_x, verbose=0)
print('RFUAV peek labels:', peek_y.numpy().tolist())
print('RFUAV peek unique labels:', sorted(set(peek_y.numpy().tolist())))
print('RFUAV peek probs finite:', bool(np.isfinite(peek_probs).all()), 'min/max:', float(np.min(peek_probs)), float(np.max(peek_probs)))
print('RFUAV peek predicted classes:', np.argmax(peek_probs, axis=1).tolist())

class ValidationMacroF1(tf.keras.callbacks.Callback):
    def __init__(self, validation_frame, max_samples=0):
        super().__init__()
        self.validation_frame = validation_frame.head(max_samples).copy() if max_samples > 0 else validation_frame.copy()
        print(f'ValidationMacroF1 samples: {len(self.validation_frame)} / {len(validation_frame)}')

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        y_true = self.validation_frame['label_idx'].to_numpy(dtype=np.int64)
        y_pred = []
        for row in self.validation_frame.itertuples(index=False):
            x = (load_precomputed_spectrogram(row.spec_path) if hasattr(row, 'spec_path') else prepare_spectrogram(row.filepath))[None, ...]
            probs = self.model.predict(x, batch_size=BATCH_SIZE, verbose=0)[0]
            y_pred.append(int(np.argmax(probs)))
        macro_f1 = float(f1_score(y_true, np.asarray(y_pred), average='macro', zero_division=0))
        logs['val_macro_f1'] = macro_f1
        print(f' - val_macro_f1: {macro_f1:.4f}')

callbacks = [
    ValidationMacroF1(val_frame_for_fit, max_samples=int(os.getenv('RFUAV_VAL_METRIC_LIMIT', RFUAV_VAL_METRIC_LIMIT_DEFAULT))),
    ModelCheckpoint(best_path, monitor='val_macro_f1', mode='max', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]

existing_history_df = pd.read_csv(history_csv_path) if history_csv_path.exists() else pd.DataFrame()
last_global_epoch = int(existing_history_df['global_epoch'].max()) if not existing_history_df.empty else 0
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_epochs = int(os.getenv('RFUAV_CNN_EPOCHS', '20'))

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=run_epochs,
    steps_per_epoch=train_steps,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1,
)

if best_path.exists():
    best_model = load_model(best_path, compile=False)
    best_model.save(final_path)
else:
    model.save(final_path)

history_payload = {
    'run_id': run_id,
    'history': history.history,
    'label_names': label_names,
    'config': {
        'input_shape': list(input_shape),
        'num_classes': int(num_classes),
        'data_fraction': float(DATA_FRACTION),
        'spec_nfft': int(SPEC_NFFT),
        'spec_hop': int(SPEC_HOP),
        'spec_time_bins': int(SPEC_TIME_BINS),
        'max_iq_samples': int(MAX_IQ_SAMPLES),
    },
}
history_path.write_text(json.dumps(history_payload, indent=2), encoding='utf-8')
run_history_path = model_dir / f'rfuav_spectrogram_cnn_history_{run_id}.json'
run_history_path.write_text(json.dumps(history_payload, indent=2), encoding='utf-8')

rows = []
for epoch_idx in range(len(history.history.get('loss', []))):
    row = {'run_id': run_id, 'epoch': epoch_idx + 1, 'global_epoch': last_global_epoch + epoch_idx + 1}
    for key, values in history.history.items():
        row[key] = values[epoch_idx]
    rows.append(row)
new_history_df = pd.DataFrame(rows)
combined_history_df = pd.concat([existing_history_df, new_history_df], ignore_index=True)
combined_history_df.to_csv(history_csv_path, index=False)
print('Saved canonical final model:', final_path)
print('Saved history:', history_path)
print('Saved cumulative history:', history_csv_path)


In [ ]:
# Cell 7: Evaluate RFUAV spectrogram CNN full-complex spectrogram model
model = load_model(final_path if final_path.exists() else best_path, compile=False)

def predict_frame(frame: pd.DataFrame):
    if EVAL_LIMIT > 0:
        frame = frame.head(EVAL_LIMIT).copy()
    path_col = 'spec_path' if 'spec_path' in frame.columns else 'filepath'
    exists_mask = frame[path_col].map(lambda value: Path(value).exists())
    if not exists_mask.all():
        print(f'Eval: dropping {int((~exists_mask).sum())} missing files before prediction.')
        frame = frame[exists_mask].reset_index(drop=True)
    probs = []
    for row in frame.itertuples(index=False):
        x = (load_precomputed_spectrogram(row.spec_path) if hasattr(row, 'spec_path') else prepare_spectrogram(row.filepath))[None, ...]
        probs.append(model.predict(x, batch_size=BATCH_SIZE, verbose=0)[0])
    return frame, np.asarray(probs, dtype=np.float32)

eval_source_df = precomputed_test_df if 'precomputed_test_df' in globals() else test_df
eval_df, probs = predict_frame(eval_source_df)
y_true = eval_df['label_idx'].to_numpy(dtype=np.int64)
y_pred = probs.argmax(axis=1)
acc = float(accuracy_score(y_true, y_pred))
macro_f1 = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
weighted_f1 = float(f1_score(y_true, y_pred, average='weighted', zero_division=0))
print('RFUAV spectrogram CNN full-complex spectrogram report')
print('accuracy:', acc)
print('macro_f1:', macro_f1)
print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

metrics = {
    'model': 'rfuav_spectrogram_cnn',
    'accuracy': acc,
    'macro_f1': macro_f1,
    'weighted_f1': weighted_f1,
    'num_classes': int(num_classes),
    'test_samples': int(len(eval_df)),
    'label_names': label_names,
}
metrics_path = outputs_dir / '34_rfuav_spectrogram_cnn_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved metrics:', metrics_path)

fig, ax = plt.subplots(figsize=(max(12, num_classes * 0.45), max(10, num_classes * 0.35)))
sns.heatmap(confusion_matrix(y_true, y_pred, labels=np.arange(num_classes)), cmap='Blues', xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_title('RFUAV spectrogram CNN Full-Complex Spectrogram Confusion Matrix')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
plt.tight_layout()
cm_path = outputs_dir / '34_rfuav_spectrogram_cnn_confusion_matrix.png'
plt.savefig(cm_path, dpi=180)
print('Saved:', cm_path)
plt.show()


In [ ]:
# Cell 8: Plot cumulative RFUAV spectrogram CNN training and validation curves
if not history_csv_path.exists():
    raise FileNotFoundError(f'Missing cumulative history file: {history_csv_path}')

history_df = pd.read_csv(history_csv_path)
print('History rows:', len(history_df))
print('Runs:', history_df['run_id'].nunique())
print(history_df.groupby('run_id').agg({'epoch': 'max', 'val_accuracy': 'max'}).tail(10))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(history_df['global_epoch'], history_df.get('accuracy', []), marker='o', label='train_accuracy')
if 'val_accuracy' in history_df:
    axes[0].plot(history_df['global_epoch'], history_df['val_accuracy'], marker='o', label='val_accuracy')
    best_idx = history_df['val_accuracy'].idxmax()
    axes[0].scatter([history_df.loc[best_idx, 'global_epoch']], [history_df.loc[best_idx, 'val_accuracy']], color='red', zorder=5, label='best_val_accuracy')
if 'val_macro_f1' in history_df:
    axes[0].plot(history_df['global_epoch'], history_df['val_macro_f1'], marker='o', label='val_macro_f1')
axes[0].set_title('RFUAV spectrogram CNN Full-Complex Spectrogram Accuracy')
axes[0].set_xlabel('Global epoch')
axes[0].set_ylabel('Score')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(history_df['global_epoch'], history_df.get('loss', []), marker='o', label='train_loss')
if 'val_loss' in history_df:
    axes[1].plot(history_df['global_epoch'], history_df['val_loss'], marker='o', label='val_loss')
axes[1].set_title('RFUAV spectrogram CNN Full-Complex Spectrogram Loss')
axes[1].set_xlabel('Global epoch')
axes[1].set_ylabel('Loss')
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout()
plot_path = outputs_dir / '34_rfuav_spectrogram_cnn_training_curves.png'
plt.savefig(plot_path, dpi=180)
print('Saved:', plot_path)
plt.show()
